In [1]:
import jax.numpy as jnp
import jax
import matplotlib.pyplot as plt

from NumericalMethods.rk45_solver import rk45
from NumericalMethods.animation import animate_pendulum
from jax.experimental.ode import odeint

from IPython.display import HTML

In [11]:
def rhs(y, t, PENDULUM_MASS, CART_MASS, LEN):
    GRAVITY = 9.81

    x = y[0]
    theta = y[1]
    v = y[2]
    omega = y[3]

    FORCE = 0

    omega_dot = (FORCE * jnp.cos(theta) + GRAVITY * CART_MASS * jnp.sin(theta) + (GRAVITY - LEN * jnp.cos(theta) * omega**2) * PENDULUM_MASS * jnp.sin(theta)) / (LEN * (CART_MASS + PENDULUM_MASS * jnp.sin(theta)**2))
    FORCE = -CART_MASS * 1 / jnp.cos(theta) * (omega_dot * LEN + GRAVITY + jnp.sin(theta)) - PENDULUM_MASS * (GRAVITY - LEN * jnp.cos(theta) * omega**2 + omega_dot * LEN * jnp.sin(theta)) * jnp.tan(theta)
    # FORCE = 1 / jnp.cos(theta) * (-omega_dot * LEN * (-CART_MASS + PENDULUM_MASS * (jnp.cos(theta) **2 - 1)) - GRAVITY * (PENDULUM_MASS + CART_MASS) * jnp.sin(theta) + LEN * PENDULUM_MASS * omega**2 * jnp.cos(theta) * jnp.sin(theta))
    v_dot = (FORCE + (GRAVITY * jnp.cos(theta) - LEN * omega**2) * PENDULUM_MASS * jnp.sin(theta)) / (CART_MASS + PENDULUM_MASS * jnp.sin(theta)**2)
    omega_dot = -omega_dot

    return jnp.array([v, omega, v_dot, omega_dot])

In [144]:
y0 = jnp.array([0, jnp.pi / 2, 0, 0])
ts = jnp.linspace(0, 10, 100)

rhs_ = lambda y, t: rhs(y, t, 1, 10, 10)
ys = rk45(rhs_, ts, y0, h0=0.001)

In [51]:
def rhs(y, t, lmbda):
    return -lmbda * y

ts = jnp.linspace(1, 10, 100)
y0 = 1.

ys_true = odeint(rhs, y0, ts, 1)

def loss(lmbda):
    ys = odeint(rhs, y0, ts, lmbda)

    return jnp.sum((ys - ys_true)**2)

In [54]:
rate = 0.2
lmbda_guess = 10.
loss_val_grad = jax.value_and_grad(loss)
for _ in range(1000):
    val, grad = loss_val_grad(lmbda_guess)
    print(val, grad, lmbda_guess)
    lmbda_guess = lmbda_guess - grad * rate


4.044964 0.12533647 10.0
4.0418143 0.1258979 9.974933
4.0386376 0.12646534 9.949753
4.0354323 0.12703896 9.924459
4.0321956 0.1276191 9.899052
4.028932 0.1282057 9.873528
4.0256357 0.12879895 9.847886
4.022311 0.12939855 9.822126
4.0189557 0.13000531 9.796247
4.0155654 0.1306192 9.770246
4.012146 0.13124003 9.744122
4.0086923 0.13186803 9.717874
4.0052066 0.13250352 9.6915
4.001688 0.1331465 9.664999
3.9981327 0.13379712 9.63837
3.9945447 0.13445537 9.61161
3.990919 0.13512194 9.58472
3.9872582 0.13579628 9.557695
3.9835603 0.13647898 9.530536
3.9798255 0.13717023 9.50324
3.976054 0.13786997 9.475805
3.9722412 0.13857853 9.448232
3.9683907 0.139296 9.420516
3.9644992 0.14002265 9.392657
3.960569 0.14075847 9.364653
3.9565942 0.14150389 9.336501
3.9525802 0.14225891 9.3082
3.9485214 0.14302418 9.279748
3.9444208 0.14379911 9.251143
3.940272 0.14458455 9.2223835
3.9360797 0.14538075 9.193466
3.9318419 0.14618753 9.16439
3.9275544 0.14700559 9.135152
3.9232218 0.14783439 9.105751
3.918836

In [12]:
args = {
    'PENDULUM_MASS': 0.4,
    'CART_MASS': 10.,
    'LEN': 1.
}

ts = jnp.linspace(0, 10, 100)
y0 = jnp.array([0, 0.1, 0, 0], dtype=jnp.float32)

d = rhs(y0, ts[0], *args.values())

ys = odeint(rhs, y0, ts, *args.values())

In [13]:
anim = animate_pendulum(ts, ys, 1.)
plt.close()
HTML(anim.to_jshtml())